In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.mathtext import _mathtext as mathtext
mathtext.FontConstantsBase.sup1 = 0.45
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

plt.rcParams['font.family'] = 'Helvetica'
plt.rcParams['font.size'] = 16
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.cal'] = 'Helvetica'
plt.rcParams['mathtext.rm'] = 'Helvetica'
plt.rcParams['mathtext.it'] = 'Helvetica:italic'
plt.rcParams['mathtext.bf'] = 'Helvetica:bold'

In [ ]:
height_data = pd.read_csv('../../data/supplementary_figures/obs/Collect/mangrove_db_Height_Collect.csv')
height_data['Height'] = pd.to_numeric(height_data['Height'], errors='coerce')

Authors = ['Lang', 'Potapov', 'Simard']
for Author in Authors:
    file_path = f'../../data/supplementary_figures/obs/mangrove_db_drivers/Height/mangrove_db_{Author}.csv'
    height_product = pd.read_csv(file_path)
    height_data[Author] = height_product['Height']

height_data = height_data.dropna()
height_data = height_data.groupby(['Lat', 'Lon']).mean().reset_index()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plt.subplots_adjust(wspace=0.25)
for i, author in enumerate(Authors):
    ax = axes[i]
    y = height_data[author]
    x = height_data['Height']

    R2 = np.corrcoef(x, y)[0, 1] ** 2
    RMSE = np.sqrt(mean_squared_error(x, y))
    MAE = np.mean(np.abs(x - y))
    PBIAS = 100 * np.sum(y - x) / np.sum(x)

    coefficients = np.polyfit(x, y, 1)
    polynomial = np.poly1d(coefficients)
    # equation = f"y = {polynomial.coefficients[0]:.2f}x + {polynomial.coefficients[1]:.2f}\n$R^2 = {R2:.2f}$\nRMSE = {RMSE:.2f} m\nPBIAS = {PBIAS:.2f}%"
    equation = f"RMSE = {RMSE:.2f} m\nMAE = {MAE:.2f}\nPBIAS = {PBIAS:.2f}%"
    ax.scatter(x, y, facecolors='none', edgecolors='gray', s=30)
    plot_range = [-2, 45]
    ax.plot(plot_range, plot_range, linestyle='--', color='gray', label="1:1 line")
    ax.plot(plot_range, polynomial(plot_range), color='red', label="Regression line")
    ax.set_xlim(plot_range)
    ax.set_ylim(plot_range)
    ax.set_ylabel(f'Canopy height from {author} et al. (m)')
    ax.set_xlabel('Canopy height collected by this study (m)')
    ax.text(0.05, 0.95, equation, transform=ax.transAxes, verticalalignment='top')
    ax.set_title(f'{chr(ord("a") + i)}', loc='left', fontweight='bold')

# plt.tight_layout()
plt.savefig('../../figures/supplementary/figS16_canopy_height_product_comparison.png', dpi=400, bbox_inches='tight')
plt.show()